Import thư viện

In [ ]:
# RF-KF | Cell 1: Import các thư viện

import os
import kagglehub
import pandas as pd
import numpy as np

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import time

Tải và đọc dữ liệu

In [ ]:
# RF-KF | Cell 2: Tải và đọc bộ dữ liệu kim cương

path = kagglehub.dataset_download("nancyalaswad90/diamonds-prices")

file_path = os.path.join(path, "Diamonds Prices2022.csv")

diamonds = pd.read_csv(file_path)

print("Kích thước dữ liệu ban đầu:", diamonds.shape)
print("\n5 dòng dữ liệu đầu tiên:")
display(diamonds.head())

Using Colab cache for faster access to the 'diamonds-prices' dataset.
Kích thước dữ liệu ban đầu: (53943, 11)

5 dòng dữ liệu đầu tiên:


,Unnamed: 0,carat,cut,color,clarity,depth,table,price,x,y,z
0,1,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,2,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,3,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,4,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,5,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


Kiểm tra dữ liệu ban đầu


In [ ]:
# RF-KF | Cell 3: Kiểm tra thông tin dữ liệu

print("Thông tin bộ dữ liệu:")
diamonds.info()

print("\nSố giá trị thiếu ở từng cột:")
print(diamonds.isnull().sum())

print("\nSố dòng trùng lặp:", diamonds.duplicated().sum())

print("\nThống kê dữ liệu:")
display(diamonds.describe())

Thông tin bộ dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53943 entries, 0 to 53942
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  53943 non-null  int64  
 1   carat       53943 non-null  float64
 2   cut         53943 non-null  object 
 3   color       53943 non-null  object 
 4   clarity     53943 non-null  object 
 5   depth       53943 non-null  float64
 6   table       53943 non-null  float64
 7   price       53943 non-null  int64  
 8   x           53943 non-null  float64
 9   y           53943 non-null  float64
 10  z           53943 non-null  float64
dtypes: float64(6), int64(2), object(3)
memory usage: 4.5+ MB

Số giá trị thiếu ở từng cột:
Unnamed: 0    0
carat         0
cut           0
color         0
clarity       0
depth         0
table         0
price         0
x             0
y             0
z             0
dtype: int64

Số dòng trùng lặp: 0

Thống kê dữ liệu:


,Unnamed: 0,carat,depth,table,price,x,y,z
count,53943.000000,53943.000000,53943.000000,53943.000000,53943.000000,53943.000000,53943.000000,53943.000000
mean,26972.000000,0.797935,61.749322,57.457251,3932.734294,5.731158,5.734526,3.538730
std,15572.147122,0.473999,1.432626,2.234549,3989.338447,1.121730,1.142103,0.705679
min,1.000000,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,13486.500000,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,26972.000000,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,40457.500000,1.040000,62.500000,59.000000,5324.000000,6.540000,6.540000,4.040000
max,53943.000000,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


Tiền xử lý dữ liệu

In [ ]:
# RF-KF | Cell 4: Tiền xử lý dữ liệu

# Xóa cột không cần thiết
diamonds = diamonds.drop(columns=["Unnamed: 0"], errors="ignore")

# Loại bỏ những mẫu có kích thước bằng 0
diamonds = diamonds[
    (diamonds["x"] > 0) &
    (diamonds["y"] > 0) &
    (diamonds["z"] > 0)
].copy()

# Xóa các dòng trùng lặp
diamonds = diamonds.drop_duplicates().reset_index(drop=True)

print("Kích thước dữ liệu sau tiền xử lý:", diamonds.shape)
print("Số dòng còn lại:", len(diamonds))

Kích thước dữ liệu sau tiền xử lý: (53775, 10)
Số dòng còn lại: 53775


Mã hóa dữ liệu phân loại

In [ ]:
# RF-KF | Cell 5: Mã hóa các thuộc tính phân loại

# Thứ tự chất lượng của các thuộc tính
cut_order = [
    "Fair", "Good", "Very Good", "Premium", "Ideal"
]

color_order = [
    "J", "I", "H", "G", "F", "E", "D"
]

clarity_order = [
    "I1", "SI2", "SI1", "VS2",
    "VS1", "VVS2", "VVS1", "IF"
]

encoder = OrdinalEncoder(
    categories=[cut_order, color_order, clarity_order]
)

diamonds[["cut", "color", "clarity"]] = encoder.fit_transform(
    diamonds[["cut", "color", "clarity"]]
)

print("Dữ liệu sau khi mã hóa:")
display(diamonds.head())

Dữ liệu sau khi mã hóa:


,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,4.0,5.0,1.0,61.5,55.0,326,3.95,3.98,2.43
1,0.21,3.0,5.0,2.0,59.8,61.0,326,3.89,3.84,2.31
2,0.23,1.0,5.0,4.0,56.9,65.0,327,4.05,4.07,2.31
3,0.29,3.0,1.0,3.0,62.4,58.0,334,4.20,4.23,2.63
4,0.31,1.0,0.0,1.0,63.3,58.0,335,4.34,4.35,2.75


Tách đặc trưng và nhãn

In [ ]:
# RF-KF | Cell 6: Tách đặc trưng X và biến mục tiêu y

X = diamonds.drop(columns=["price"])
y = diamonds["price"]

print("Số mẫu:", X.shape[0])
print("Số thuộc tính:", X.shape[1])
print("Biến mục tiêu: price")

Số mẫu: 53775
Số thuộc tính: 9
Biến mục tiêu: price


khởi tạo 5-Fold Cross-Validation và thiết lập tham số Random Forest.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np
import time

# Khởi tạo phương pháp 5-Fold
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Bộ tham số tốt nhất từ phần Hold-out
best_params = {
    "n_estimators": 150,
    "max_depth": 15,
    "min_samples_split": 2,
    "min_samples_leaf": 2
}

# Danh sách lưu kết quả
ket_qua_kfold = []

print("Số mẫu đưa vào 5-Fold:", len(X_train))
print("Bộ tham số sử dụng:", best_params)

Số mẫu đưa vào 5-Fold: 43020
Bộ tham số sử dụng: {'n_estimators': 150, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 2}


chạy vòng lặp huấn luyện và đánh giá Random Forest qua 5-Fold.


In [ ]:
# Duyệt lần lượt qua 5 Fold
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), start=1):

    # Lấy dữ liệu Train của Fold hiện tại
    X_train_fold = X_train.iloc[train_idx]
    y_train_fold = y_train.iloc[train_idx]

    # Lấy dữ liệu Validation của Fold hiện tại
    X_val_fold = X_train.iloc[val_idx]
    y_val_fold = y_train.iloc[val_idx]

    # Khởi tạo mô hình Random Forest
    model = RandomForestRegressor(
        **best_params,
        criterion="squared_error",
        random_state=42,
        n_jobs=-1
    )

    # Bắt đầu đo thời gian
    start_time = time.perf_counter()

    # Huấn luyện mô hình
    model.fit(X_train_fold, y_train_fold)

    # Tính thời gian huấn luyện
    train_time = time.perf_counter() - start_time

    # Dự đoán trên Train
    y_pred_train = model.predict(X_train_fold)

    # Dự đoán trên Validation
    y_pred_val = model.predict(X_val_fold)

    # Tính kết quả cho hai tập
    for ten_tap, y_true, y_pred in [
        ("train", y_train_fold, y_pred_train),
        ("test", y_val_fold, y_pred_val)
    ]:

        # Tính MAE
        mae = mean_absolute_error(y_true, y_pred)

        # Tính MSE
        mse = mean_squared_error(y_true, y_pred)

        # Tính RMSE
        rmse = np.sqrt(mse)

        # Tính R bình phương
        r2 = r2_score(y_true, y_pred)

        # Lưu kết quả
        ket_qua_kfold.append({
            "Fold": fold,
            "Tập dữ liệu": ten_tap,
            "Số mẫu": len(y_true),
            "MAE (USD)": mae,
            "MSE (USD²)": mse,
            "RMSE (USD)": rmse,
            "R²": r2,
            "Thời gian huấn luyện (giây)": (
                train_time if ten_tap == "train" else np.nan
            )
        })

    print(f"Đã hoàn thành Fold {fold}/5")

# Chuyển kết quả thành bảng
df_kfold = pd.DataFrame(ket_qua_kfold)

print("\nKẾT QUẢ 5-FOLD RANDOM FOREST")
display(df_kfold.round(4).fillna(""))

Đã hoàn thành Fold 1/5
Đã hoàn thành Fold 2/5
Đã hoàn thành Fold 3/5
Đã hoàn thành Fold 4/5
Đã hoàn thành Fold 5/5

KẾT QUẢ 5-FOLD RANDOM FOREST


,Fold,Tập dữ liệu,Số mẫu,MAE (USD),MSE (USD²),RMSE (USD),R²,Thời gian huấn luyện (giây)
0,1,train,34416,157.4470,95577.1061,309.1555,0.9940,23.946
1,1,test,8604,274.8253,319955.8561,565.6464,0.9801,
2,2,train,34416,159.2924,100448.7826,316.9366,0.9937,18.9884
3,2,test,8604,270.3439,287242.3244,535.9499,0.9818,
4,3,train,34416,158.2483,99591.8509,315.5818,0.9937,19.8493
5,3,test,8604,272.7603,310260.9484,557.0107,0.9806,
6,4,train,34416,159.3608,98827.7490,314.3688,0.9939,21.2024
7,4,test,8604,262.9537,282067.7219,531.1005,0.9815,
8,5,train,34416,157.7304,98289.0639,313.5109,0.9938,19.7234
9,5,test,8604,276.8987,308884.3429,555.7736,0.9812,


Tính trung bình và tổng hợp kết quả 5-Fold

In [ ]:
# Các cột cần tính trung bình
cot_chi_so = [
    "MAE (USD)",
    "MSE (USD²)",
    "RMSE (USD)",
    "R²"
]

# Tạo danh sách lưu hai dòng trung bình
dong_trung_binh = []

# Tính trung bình riêng cho Train và Validation
for ten_tap in ["train", "test"]:

    # Lọc dữ liệu theo loại tập
    df_tap = df_kfold[df_kfold["Tập dữ liệu"] == ten_tap]

    # Tạo dòng trung bình
    dong = {
        "Fold": "Trung bình",
        "Tập dữ liệu": ten_tap,
        "Số mẫu": df_tap["Số mẫu"].mean()
    }

    # Tính trung bình các chỉ số
    for cot in cot_chi_so:
        dong[cot] = df_tap[cot].mean()

    # Chỉ ghi thời gian ở dòng Train
    if ten_tap == "train":
        dong["Thời gian huấn luyện (giây)"] = df_tap[
            "Thời gian huấn luyện (giây)"
        ].mean()
    else:
        dong["Thời gian huấn luyện (giây)"] = np.nan

    dong_trung_binh.append(dong)

# Ghép kết quả 5 Fold và hai dòng trung bình
bang_kfold = pd.concat(
    [df_kfold, pd.DataFrame(dong_trung_binh)],
    ignore_index=True
)

# Tạo bảng hiển thị
bang_hien_thi = bang_kfold.round(4).fillna("")

# Hiển thị
print("KẾT QUẢ HUẤN LUYỆN TƯƠNG ỨNG VỚI BỘ THAM SỐ TRÊN:")
display(bang_hien_thi)

KẾT QUẢ HUẤN LUYỆN TƯƠNG ỨNG VỚI BỘ THAM SỐ TRÊN:


,Fold,Tập dữ liệu,Số mẫu,MAE (USD),MSE (USD²),RMSE (USD),R²,Thời gian huấn luyện (giây)
0,1,train,34416.0,157.4470,95577.1061,309.1555,0.9940,23.946
1,1,test,8604.0,274.8253,319955.8561,565.6464,0.9801,
2,2,train,34416.0,159.2924,100448.7826,316.9366,0.9937,18.9884
3,2,test,8604.0,270.3439,287242.3244,535.9499,0.9818,
4,3,train,34416.0,158.2483,99591.8509,315.5818,0.9937,19.8493
5,3,test,8604.0,272.7603,310260.9484,557.0107,0.9806,
6,4,train,34416.0,159.3608,98827.7490,314.3688,0.9939,21.2024
7,4,test,8604.0,262.9537,282067.7219,531.1005,0.9815,
8,5,train,34416.0,157.7304,98289.0639,313.5109,0.9938,19.7234
9,5,test,8604.0,276.8987,308884.3429,555.7736,0.9812,


Tổng hợp tham số và xuất kết quả 5-Fold ra Excel

In [ ]:
from google.colab import files

# Bảng tham số
bang_tham_so = pd.DataFrame({
    "Tham số": [
        "n_estimators",
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "Thời gian huấn luyện trung bình"
    ],
    "Giá trị": [
        150,
        15,
        2,
        2,
        f"{df_kfold[df_kfold['Tập dữ liệu'] == 'train']['Thời gian huấn luyện (giây)'].mean():.4f} giây"
    ]
})

# Hiển thị bảng tham số
print("BỘ THAM SỐ SỬ DỤNG")
display(bang_tham_so)

# Lưu cả hai bảng trong một file Excel
with pd.ExcelWriter("Random_Forest_KFold.xlsx") as writer:
    bang_tham_so.to_excel(
        writer,
        sheet_name="Tham so",
        index=False
    )

    bang_hien_thi.to_excel(
        writer,
        sheet_name="Ket qua 5 Fold",
        index=False
    )

# Tải file về máy
files.download("Random_Forest_KFold.xlsx")

BỘ THAM SỐ SỬ DỤNG


,Tham số,Giá trị
0,n_estimators,150
1,max_depth,15
2,min_samples_split,2
3,min_samples_leaf,2
4,Thời gian huấn luyện trung bình,20.7419 giây


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Chạy toàn bộ 5-Fold, tổng hợp kết quả và xuất Excel


In [ ]:

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display
import pandas as pd
import numpy as np
import time

# Bộ tham số tốt nhất từ Hold-out
best_params = {
    "n_estimators": 150,
    "max_depth": 15,
    "min_samples_split": 2,
    "min_samples_leaf": 2
}

# Chia 5-Fold trên toàn bộ dữ liệu
kf = KFold(n_splits=5, shuffle=True, random_state=42)

ket_qua_kfold = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):

    X_tr = X.iloc[train_idx]
    X_te = X.iloc[test_idx]
    y_tr = y.iloc[train_idx]
    y_te = y.iloc[test_idx]

    model = RandomForestRegressor(
        **best_params,
        criterion="squared_error",
        random_state=42,
        n_jobs=-1
    )

    start_time = time.perf_counter()
    model.fit(X_tr, y_tr)
    train_time = time.perf_counter() - start_time

    # Dự đoán trên Train và Test của fold
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)

    for ten_tap, y_true, y_pred in [
        ("train", y_tr, y_pred_train),
        ("test", y_te, y_pred_test)
    ]:
        mse = mean_squared_error(y_true, y_pred)

        ket_qua_kfold.append({
            "Fold": fold,
            "Tập dữ liệu": ten_tap,
            "Số mẫu": len(y_true),
            "MAE (USD)": mean_absolute_error(y_true, y_pred),
            "MSE (USD²)": mse,
            "RMSE (USD)": np.sqrt(mse),
            "R²": r2_score(y_true, y_pred),
            "Thời gian huấn luyện (giây)": (
                train_time if ten_tap == "train" else np.nan
            )
        })

    print(f"Đã hoàn thành Fold {fold}/5")

# Tạo bảng kết quả
df_kfold = pd.DataFrame(ket_qua_kfold)

# Tính trung bình theo tập Train và Test
cot_trung_binh = [
    "Số mẫu",
    "MAE (USD)",
    "MSE (USD²)",
    "RMSE (USD)",
    "R²",
    "Thời gian huấn luyện (giây)"
]

trung_binh = (
    df_kfold.groupby("Tập dữ liệu", sort=False)[cot_trung_binh]
    .mean()
    .reset_index()
)

trung_binh.insert(0, "Fold", "Trung bình")

# Ghép bảng 5 fold và trung bình
bang_kfold = pd.concat(
    [df_kfold, trung_binh],
    ignore_index=True
)

print("\nBỘ THAM SỐ SỬ DỤNG:")
display(pd.DataFrame(
    list(best_params.items()),
    columns=["Tham số", "Giá trị"]
))

print("\nKẾT QUẢ HUẤN LUYỆN THEO 5-FOLD:")
display(bang_kfold.round(4).fillna(""))

# Xuất file Excel
bang_kfold.to_excel(
    "Random_Forest_KFold.xlsx",
    index=False
)

from google.colab import files
files.download("Random_Forest_KFold.xlsx")


Đã hoàn thành Fold 1/5
Đã hoàn thành Fold 2/5
Đã hoàn thành Fold 3/5
Đã hoàn thành Fold 4/5
Đã hoàn thành Fold 5/5

BỘ THAM SỐ SỬ DỤNG:


,Tham số,Giá trị
0,n_estimators,150
1,max_depth,15
2,min_samples_split,2
3,min_samples_leaf,2



KẾT QUẢ HUẤN LUYỆN THEO 5-FOLD:


,Fold,Tập dữ liệu,Số mẫu,MAE (USD),MSE (USD²),RMSE (USD),R²,Thời gian huấn luyện (giây)
0,1,train,43020.0,161.6494,102795.0726,320.6167,0.9935,33.3439
1,1,test,10755.0,266.8492,272006.6520,521.5426,0.9828,
2,2,train,43020.0,161.0189,99854.6540,315.9979,0.9937,22.2588
3,2,test,10755.0,265.5080,283446.7788,532.3972,0.9818,
4,3,train,43020.0,162.0162,100283.2598,316.6753,0.9937,25.1924
5,3,test,10755.0,262.6772,283926.0671,532.8471,0.9821,
6,4,train,43020.0,162.5760,103152.6115,321.1738,0.9935,30.3464
7,4,test,10755.0,267.7982,284688.1451,533.5618,0.9825,
8,5,train,43020.0,159.8963,98087.6491,313.1895,0.9938,25.5289
9,5,test,10755.0,269.7564,320451.5749,566.0844,0.9799,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>